# TDE 04: Evolução dos modelos e modelo final
## Análise de Volatilidade de Ações (Optiver Realized Volatility Prediction)

**Disciplina:** Inteligência Artificial, UniFAP 2026.2
**Professor:** Alexsandro Oliveira Alexandrino
**Grupo:** João Gabriel Oliveira Silva e Lemuel Lopes Sousa
**Etapas anteriores:** TDE 01 (análise exploratória) e TDE 03 (baseline)

### O que este notebook entrega

| Pedido do TDE 04 | Onde está |
|---|---|
| Evoluções em relação ao baseline | Seções 5 a 8 (novos atributos, alvo em log, ajuste de hiperparâmetros) |
| SVM | Seção 9.3 (SVR com kernel RBF) |
| Redes neurais | Seção 9.4 (MLP, perceptron multicamadas) |
| Modelos não estudados em sala | Seção 9.5 (LightGBM, XGBoost, Extra Trees) |
| Comparação dos modelos | Seção 10 |
| Modelo final | Seções 11 e 12 |
| Análise de erros e dificuldades | Seções 13 e 14 |

## 1. Onde paramos no TDE 03

O problema: prever a **volatilidade realizada dos próximos 10 minutos** (`target`) de 112 ações, a partir do que aconteceu nos 10 minutos anteriores no livro de ofertas (*order book*) e nos negócios (*trades*).

No TDE 03 montamos o baseline:

- 9 atributos simples (volatilidade do WAP, spread, volumes, contagens);
- divisão 80/20 por `time_id`, para que uma mesma janela de tempo nunca apareça no treino e no teste ao mesmo tempo;
- Regressão Linear, Árvore de Decisão e Random Forest com os valores padrão;
- métricas MAE, MSE, RMSE, R² e RMSPE (a métrica oficial da competição).

O próprio TDE 03 listou as limitações que este TDE ataca:

1. poucos atributos e nenhum por sub-janela de tempo ou do segundo nível do book;
2. `stock_id` e o efeito de mercado (várias ações voláteis ao mesmo tempo) não aproveitados;
3. nenhum ajuste de hiperparâmetros;
4. modelos treinados com erro quadrático no alvo bruto, enquanto a competição mede erro percentual.

Para a comparação ser justa, este notebook **reproduz o baseline do TDE 03 com a mesma divisão de treino e teste** (`random_state=42`) e avalia tudo no mesmo conjunto de teste.

## 2. Preparação do ambiente

Rode no **Google Colab**, célula por célula, de cima para baixo.

1. Entre no Kaggle e aceite as regras da competição **Optiver Realized Volatility Prediction**.
2. Gere o `kaggle.json` (Kaggle → *Settings* → *API* → *Create New Token*).
3. Rode a célula de download e envie o `kaggle.json` quando o Colab pedir.

Tempo total estimado no Colab gratuito: de 30 a 45 minutos. A maior parte é a extração de atributos (os arquivos Parquet têm cerca de 3 GB) e o treino do Random Forest e do SVR. Se o tempo estiver curto, mude `N_ACOES_MAX` na célula de configuração para, por exemplo, `40`.

In [ ]:
%pip -q install kaggle pyarrow lightgbm xgboost

In [ ]:
# Pasta de dados e download (só baixa se ainda não existir)
import os, glob, zipfile, shutil, subprocess

DATA_DIR = os.environ.get("OPTIVER_DIR", "/content/optiver")
os.makedirs(DATA_DIR, exist_ok=True)

def encontrar(padrao):
    return glob.glob(os.path.join(DATA_DIR, "**", padrao), recursive=True)

if encontrar("train.csv"):
    print("Dados já disponíveis em", DATA_DIR, "- download ignorado.")
else:
    kaggle_dir = os.path.expanduser("~/.kaggle")
    os.makedirs(kaggle_dir, exist_ok=True)
    kaggle_json_path = os.path.join(kaggle_dir, "kaggle.json")
    try:
        from google.colab import files
        if not os.path.exists(kaggle_json_path):
            uploaded = files.upload()  # selecione o kaggle.json
            nomes = [n for n in uploaded if n.lower().endswith(".json")]
            if not nomes:
                raise FileNotFoundError("Envie o arquivo kaggle.json gerado na sua conta do Kaggle.")
            shutil.copy(nomes[0], kaggle_json_path)  # aceita "kaggle (1).json" também
    except ImportError:
        if not os.path.exists(kaggle_json_path) and not os.environ.get("KAGGLE_KEY"):
            raise FileNotFoundError(f"Fora do Colab: coloque o kaggle.json em {kaggle_json_path}.")
    if os.path.exists(kaggle_json_path):
        os.chmod(kaggle_json_path, 0o600)

    subprocess.run(["kaggle", "competitions", "download", "-c",
                    "optiver-realized-volatility-prediction", "-p", DATA_DIR], check=True)
    for z in glob.glob(os.path.join(DATA_DIR, "*.zip")):
        with zipfile.ZipFile(z, "r") as zf:
            zf.extractall(DATA_DIR)
        os.remove(z)
    print("Download e extração concluídos.")

## 3. Bibliotecas e configuração

In [ ]:
import time, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from joblib import Parallel, delayed

from sklearn.model_selection import train_test_split, GroupKFold, RandomizedSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, make_scorer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor
import lightgbm as lgb
import xgboost as xgb

warnings.filterwarnings("ignore", category=UserWarning)

SEED = 42
np.random.seed(SEED)

# Configurações de tempo de execução
N_ACOES_MAX = None      # None = todas as ações; um número (ex.: 40) acelera o teste
AMOSTRA_BUSCA = 80_000  # linhas de treino usadas na busca de hiperparâmetros
AMOSTRA_SVR = 30_000    # o SVR cresce de forma quadrática com o número de linhas
N_FOLDS = 3             # validação cruzada por grupos de time_id

PASTA_FIG = "figuras_tde04"
os.makedirs(PASTA_FIG, exist_ok=True)

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.6f}")

plt.rcParams.update({
    "figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
    "axes.titleweight": "bold", "axes.titlesize": 12, "font.size": 10,
})
COR = {"Referência": "#9AA0A6", "Baseline TDE 03": "#7F8FA6",
       "Visto em sala": "#2E86AB", "Não visto em sala": "#E07A5F", "Modelo final": "#2A9D8F"}

def salvar(nome):
    plt.savefig(os.path.join(PASTA_FIG, nome), dpi=160, bbox_inches="tight")

print("lightgbm", lgb.__version__, "| xgboost", xgb.__version__)

## 4. Carregamento do conjunto principal

In [ ]:
train = pd.read_csv(encontrar("train.csv")[0])
print(f"Linhas: {len(train):,} | Ações: {train['stock_id'].nunique()} | Janelas (time_id): {train['time_id'].nunique():,}")
display(train["target"].describe().to_frame("target").T)

## 5. Evolução 1: novos atributos

O baseline usava 9 atributos da janela inteira. Agora calculamos, para cada ação e janela de 10 minutos:

| Grupo | Atributos | Por quê |
|---|---|---|
| Volatilidade em sub-janelas | `rv_wap1` na janela toda e nos últimos 450, 300 e 150 s | A volatilidade é persistente: o que aconteceu no fim da janela pesa mais para os próximos minutos |
| Segundo nível do book | `wap2`, `rv_wap2` | O WAP do nível 2 mostra a pressão de compra e venda que ainda não chegou ao melhor preço |
| Liquidez | spread relativo (`ask1/bid1 - 1`), diferença entre WAP1 e WAP2, desequilíbrio de volume entre compra e venda, profundidade | Ações menos líquidas oscilam mais |
| Negócios | `rv_trade` total e dos últimos 300 s, volume, ordens, número de negócios, tamanho médio | Atividade de negociação acompanha os picos de volatilidade |
| Contexto de mercado | média de `rv_wap1` de todas as ações na mesma janela | O TDE 01 mostrou que várias ações ficam voláteis ao mesmo tempo |
| Perfil da ação | média de `rv_wap1` da ação calculada **só nas janelas de treino**, e a razão entre a janela atual e essa média | Substitui o `stock_id` numérico, que não tem ordem, por uma medida com significado |

Os 9 atributos do baseline continuam na tabela, com os mesmos nomes, para reproduzir o TDE 03 exatamente.

In [ ]:
def stock_id_do_caminho(caminho):
    return int(os.path.basename(caminho.rstrip("/")).split("=")[1])


def rv(serie_r2):
    return np.sqrt(serie_r2)


def atributos_book(caminho):
    b = pd.read_parquet(caminho).sort_values(["time_id", "seconds_in_bucket"])
    b["wap1"] = (b["bid_price1"] * b["ask_size1"] + b["ask_price1"] * b["bid_size1"]) / (b["bid_size1"] + b["ask_size1"])
    b["wap2"] = (b["bid_price2"] * b["ask_size2"] + b["ask_price2"] * b["bid_size2"]) / (b["bid_size2"] + b["ask_size2"])
    for w in ("wap1", "wap2"):
        b[f"r2_{w}"] = np.log(b[w]).groupby(b["time_id"]).diff() ** 2
    b["spread"] = b["ask_price1"] - b["bid_price1"]
    b["spread_rel"] = b["ask_price1"] / b["bid_price1"] - 1
    b["wap_balance"] = (b["wap1"] - b["wap2"]).abs()
    b["desequilibrio"] = (b["bid_size1"] + b["bid_size2"] - b["ask_size1"] - b["ask_size2"]) / (
        b["bid_size1"] + b["bid_size2"] + b["ask_size1"] + b["ask_size2"])
    b["volume_total"] = b["bid_size1"] + b["ask_size1"] + b["bid_size2"] + b["ask_size2"]

    g = b.groupby("time_id")
    f = pd.DataFrame({
        # atributos do baseline (mesmos nomes do TDE 03)
        "rv_wap": rv(g["r2_wap1"].sum()),
        "spread_medio": g["spread"].mean(),
        "volume_medio_book": g["volume_total"].mean(),
        "n_atualizacoes_book": g.size(),
        # novos atributos
        "rv_wap2": rv(g["r2_wap2"].sum()),
        "spread_rel_medio": g["spread_rel"].mean(),
        "spread_rel_desvio": g["spread_rel"].std(),
        "wap_balance_medio": g["wap_balance"].mean(),
        "desequilibrio_medio": g["desequilibrio"].mean(),
    })
    for ini in (150, 300, 450):
        parte = b[b["seconds_in_bucket"] >= ini].groupby("time_id")
        f[f"rv_wap_ultimos_{600 - ini}s"] = rv(parte["r2_wap1"].sum()).reindex(f.index).fillna(0)
        if ini == 300:
            f["rv_wap2_ultimos_300s"] = rv(parte["r2_wap2"].sum()).reindex(f.index).fillna(0)
            f["n_atualizacoes_ultimos_300s"] = parte.size().reindex(f.index).fillna(0)
    return f


def atributos_trade(caminho):
    t = pd.read_parquet(caminho).sort_values(["time_id", "seconds_in_bucket"])
    t["r2"] = np.log(t["price"]).groupby(t["time_id"]).diff() ** 2
    g = t.groupby("time_id")
    f = pd.DataFrame({
        "rv_trade": rv(g["r2"].sum()),
        "volume_trades": g["size"].sum(),
        "ordens_trades": g["order_count"].sum(),
        "n_trades": g.size(),
        "tamanho_medio_trade": g["size"].mean(),
    })
    fim = t[t["seconds_in_bucket"] >= 300].groupby("time_id")
    f["rv_trade_ultimos_300s"] = rv(fim["r2"].sum()).reindex(f.index).fillna(0)
    f["n_trades_ultimos_300s"] = fim.size().reindex(f.index).fillna(0)
    return f


COLS_TRADE = ["rv_trade", "volume_trades", "ordens_trades", "n_trades", "tamanho_medio_trade",
              "rv_trade_ultimos_300s", "n_trades_ultimos_300s"]


def atributos_acao(caminho_book):
    f = atributos_book(caminho_book)
    caminho_trade = caminho_book.replace("book_train.parquet", "trade_train.parquet")
    if os.path.exists(caminho_trade):
        f = f.join(atributos_trade(caminho_trade), how="left")
    for c in COLS_TRADE:
        if c not in f:
            f[c] = np.nan
    f[COLS_TRADE] = f[COLS_TRADE].fillna(0)  # janela sem negócios
    f = f.reset_index()
    f.insert(0, "stock_id", stock_id_do_caminho(caminho_book))
    return f

In [ ]:
CACHE = os.path.join(DATA_DIR, "features_tde04.parquet")

if os.path.exists(CACHE):
    features = pd.read_parquet(CACHE)
    print("Atributos carregados do cache:", CACHE)
else:
    pastas = sorted(p for p in glob.glob(os.path.join(DATA_DIR, "**", "book_train.parquet", "stock_id=*"), recursive=True)
                    if os.path.isdir(p))
    if N_ACOES_MAX:
        pastas = pastas[:N_ACOES_MAX]
    print("Ações a processar:", len(pastas))
    inicio = time.time()
    features = pd.concat(Parallel(n_jobs=-1, verbose=2)(delayed(atributos_acao)(p) for p in pastas), ignore_index=True)
    features.to_parquet(CACHE, index=False)
    print(f"Atributos calculados em {(time.time() - inicio) / 60:.1f} min")

dados = train.merge(features, on=["stock_id", "time_id"], how="inner")

# Contexto de mercado: média da volatilidade de todas as ações na mesma janela.
# Todas as ações de uma janela são observadas no mesmo intervalo, então isso não usa informação do futuro.
dados["rv_wap_media_janela"] = dados.groupby("time_id")["rv_wap"].transform("mean")
dados["rv_wap_ultimos_300s_media_janela"] = dados.groupby("time_id")["rv_wap_ultimos_300s"].transform("mean")
dados["rv_wap_rel_janela"] = dados["rv_wap"] / dados["rv_wap_media_janela"]

print(f"Linhas após a junção: {len(dados):,} ({len(dados) / len(train) * 100:.1f}% do train.csv)")
print("Valores ausentes:", int(dados.isna().sum().sum()))
dados = dados.fillna(0)

## 6. Mesma divisão de treino e teste do TDE 03

Usamos exatamente o mesmo sorteio de `time_id` do baseline (80/20, `random_state=42`). Assim, qualquer ganho que aparecer vem do modelo e dos atributos, e não de um conjunto de teste mais fácil.

Os atributos de perfil da ação são calculados **depois** da divisão, só com as janelas de treino.

In [ ]:
time_ids = dados["time_id"].unique()
tid_treino, tid_teste = train_test_split(time_ids, test_size=0.20, random_state=SEED)
mask = dados["time_id"].isin(tid_treino)

# Perfil da ação usando só o treino (evita vazamento)
perfil = dados[mask].groupby("stock_id").agg(rv_wap_media_acao=("rv_wap", "mean"),
                                             spread_rel_medio_acao=("spread_rel_medio", "mean"))
dados = dados.drop(columns=[c for c in perfil.columns if c in dados]).merge(perfil, on="stock_id", how="left")
dados[perfil.columns] = dados[perfil.columns].fillna(dados.loc[mask, perfil.columns[0]].mean())
dados["rv_wap_rel_acao"] = dados["rv_wap"] / dados["rv_wap_media_acao"]

mask = dados["time_id"].isin(tid_treino).to_numpy()
treino, teste = dados[mask].reset_index(drop=True), dados[~mask].reset_index(drop=True)

ATRIB_BASE = ["rv_wap", "rv_wap_ultimos_300s", "spread_medio", "volume_medio_book", "n_atualizacoes_book",
              "rv_trade", "volume_trades", "ordens_trades", "n_trades"]
ATRIB_NOVOS = [c for c in dados.columns if c not in ATRIB_BASE + ["stock_id", "time_id", "target"]]
ATRIB = ATRIB_BASE + ATRIB_NOVOS

y_train, y_test = treino["target"].to_numpy(), teste["target"].to_numpy()
grupos = treino["time_id"].to_numpy()

print(f"Treino: {len(treino):,} linhas ({len(tid_treino):,} janelas) | Teste: {len(teste):,} linhas ({len(tid_teste):,} janelas)")
print("Janelas em comum:", len(set(treino["time_id"]) & set(teste["time_id"])))
print(f"Atributos: {len(ATRIB_BASE)} do baseline + {len(ATRIB_NOVOS)} novos = {len(ATRIB)}")

## 7. Evolução 2: alvo e atributos em escala logarítmica

O TDE 01 mostrou que a volatilidade tem **assimetria forte à direita**: muitas janelas calmas e poucas muito voláteis. Treinar com o alvo bruto faz o modelo se preocupar quase só com as janelas extremas.

Por isso, todos os modelos do TDE 04 aprendem `log(target)` e a previsão volta para a escala original com `exp`. Errar 0,001 numa janela de volatilidade 0,002 passa a pesar tanto quanto errar 0,01 numa janela de 0,02. É o mesmo raciocínio do RMSPE, a métrica oficial da competição.

Os atributos positivos e assimétricos também passam por `log`. Para as árvores isso não muda nada (elas só olham a ordem dos valores), mas para Regressão Linear, SVM e redes neurais faz muita diferença.

In [ ]:
NAO_LOG = {"desequilibrio_medio"}  # pode ser negativo

def transformar(df):
    X = df[ATRIB].astype(float).copy()
    for c in ATRIB:
        if c not in NAO_LOG:
            X[c] = np.log(X[c].clip(lower=0) + 1e-8)
    return X

X_train, X_test = transformar(treino), transformar(teste)
ylog_train = np.log(y_train)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
axes[0].hist(y_train, bins=120, color=COR["Visto em sala"])
axes[0].set_title("Alvo original: assimetria à direita")
axes[1].hist(ylog_train, bins=120, color=COR["Modelo final"])
axes[1].set_title("log(alvo): distribuição quase simétrica")
for ax in axes: ax.set_ylabel("Frequência")
plt.tight_layout(); salvar("01_alvo_log.png"); plt.show()

## 8. Métricas e protocolo de avaliação

As mesmas métricas do TDE 03: **MAE** (métrica principal, na unidade da volatilidade), MSE, RMSE, R² e **RMSPE** (erro percentual da competição).

Protocolo:

1. **Busca de hiperparâmetros** com `RandomizedSearchCV` e validação cruzada `GroupKFold` por `time_id` (3 dobras), só no treino. Assim a escolha não "espia" o teste.
2. Refit do melhor modelo no treino inteiro.
3. Avaliação **uma única vez** no conjunto de teste.

In [ ]:
def rmspe(y, p):
    y, p = np.asarray(y), np.asarray(p)
    return float(np.sqrt(np.mean(((y - p) / y) ** 2)))

def avaliar(y, p):
    mse = mean_squared_error(y, p)
    return {"MAE": mean_absolute_error(y, p), "MSE": mse, "RMSE": np.sqrt(mse),
            "R2": r2_score(y, p), "RMSPE": rmspe(y, p)}

# Scorer da busca: modelo prevê log, mas o erro é medido na escala original (MAE)
mae_escala_original = make_scorer(lambda yl, pl: mean_absolute_error(np.exp(yl), np.exp(pl)), greater_is_better=False)

rng = np.random.RandomState(SEED)
idx_busca = rng.choice(len(X_train), size=min(AMOSTRA_BUSCA, len(X_train)), replace=False)
cv = GroupKFold(n_splits=N_FOLDS)

resultados, previsoes, info = [], {}, {}

def registrar(nome, categoria, pred_teste, pred_treino=None, tempo=None, params="", cv_mae=None):
    previsoes[nome] = pred_teste
    linha = {"modelo": nome, "categoria": categoria, **avaliar(y_test, pred_teste)}
    linha["MAE treino"] = mean_absolute_error(y_train, pred_treino) if pred_treino is not None else np.nan
    linha["MAE validação (CV)"] = cv_mae if cv_mae is not None else np.nan
    linha["tempo de treino (s)"] = tempo if tempo is not None else np.nan
    linha["hiperparâmetros"] = params
    resultados.append(linha)
    print(f"{nome:<32} MAE teste {linha['MAE']:.6f} | RMSPE {linha['RMSPE']:.4f} | R² {linha['R2']:.4f}")

### 8.1 Referências e baseline do TDE 03 (reproduzidos)

Mesmos 9 atributos, alvo bruto e valores padrão do TDE 03.

In [ ]:
Xb_train, Xb_test = treino[ATRIB_BASE], teste[ATRIB_BASE]

dummy = DummyRegressor().fit(Xb_train, y_train)
registrar("Referência: média do treino", "Referência", dummy.predict(Xb_test), dummy.predict(Xb_train))
registrar("Referência: persistência", "Referência", teste["rv_wap"].to_numpy(), treino["rv_wap"].to_numpy())

baseline = {
    "TDE 03: Regressão Linear": make_pipeline(StandardScaler(), LinearRegression()),
    "TDE 03: Árvore de Decisão": DecisionTreeRegressor(random_state=SEED),
    "TDE 03: Random Forest": RandomForestRegressor(n_estimators=100, random_state=SEED, n_jobs=-1),
}
for nome, m in baseline.items():
    t0 = time.time(); m.fit(Xb_train, y_train); dt = time.time() - t0
    registrar(nome, "Baseline TDE 03", m.predict(Xb_test), m.predict(Xb_train), dt, "valores padrão (TDE 03)")

MELHOR_BASELINE = min(baseline, key=lambda n: mean_absolute_error(y_test, previsoes[n]))
print("\nMelhor baseline do TDE 03:", MELHOR_BASELINE)

## 9. Evolução 3: modelos ajustados

Função comum: busca aleatória de hiperparâmetros na amostra de treino, com validação por grupos de `time_id`, refit no treino inteiro e avaliação no teste.

In [ ]:
def ajustar(nome, categoria, estimador, espaco, n_iter=8, amostra=None, n_jobs_busca=1):
    idx = idx_busca if amostra is None else rng.choice(len(X_train), size=min(amostra, len(X_train)), replace=False)
    t0 = time.time()
    if espaco:
        busca = RandomizedSearchCV(estimador, espaco, n_iter=n_iter, cv=cv, scoring=mae_escala_original,
                                   random_state=SEED, n_jobs=n_jobs_busca, refit=False)
        busca.fit(X_train.iloc[idx], ylog_train[idx], groups=grupos[idx])
        melhores, cv_mae = busca.best_params_, -busca.best_score_
        estimador.set_params(**melhores)
    else:
        melhores, cv_mae = {}, None
    t_busca = time.time() - t0

    # refit: no treino inteiro, exceto quando o modelo pede amostra (SVR)
    idx_fit = np.arange(len(X_train)) if amostra is None else idx
    t0 = time.time()
    estimador.fit(X_train.iloc[idx_fit], ylog_train[idx_fit])
    t_fit = time.time() - t0

    pred_teste = np.exp(estimador.predict(X_test))
    pred_treino = np.exp(estimador.predict(X_train.iloc[idx_busca]))
    info[nome] = {"modelo": estimador, "busca_s": t_busca}
    params_txt = ", ".join(f"{k.split('__')[-1]}={v}" for k, v in melhores.items()) or "padrão"
    registrar(nome, categoria, pred_teste, None, t_fit, params_txt, cv_mae)
    resultados[-1]["MAE treino"] = mean_absolute_error(y_train[idx_busca], pred_treino)
    return estimador

### 9.1 Regressão Linear regularizada (Ridge)

A mesma ideia da Regressão Linear do baseline, agora com os novos atributos em log e regularização L2 (`alpha`) para lidar com atributos muito correlacionados entre si.

In [ ]:
ajustar("Ridge (linear regularizada)", "Visto em sala",
        make_pipeline(StandardScaler(), Ridge()),
        {"ridge__alpha": [0.01, 0.1, 1, 10, 100]}, n_iter=5)

### 9.2 Árvore de Decisão e Random Forest ajustados

No baseline a árvore cresceu sem limite e decorou o treino (overfitting). Aqui limitamos `max_depth` e `min_samples_leaf`. No Random Forest, ajustamos também `max_features` e usamos `max_samples` para cada árvore ver só parte dos dados, o que reduz o tempo e a correlação entre as árvores.

In [ ]:
ajustar("Árvore de Decisão ajustada", "Visto em sala",
        DecisionTreeRegressor(random_state=SEED),
        {"max_depth": [6, 8, 10, 12, 15], "min_samples_leaf": [20, 50, 100, 200, 400]}, n_iter=10)

ajustar("Random Forest ajustado", "Visto em sala",
        RandomForestRegressor(n_estimators=200, random_state=SEED, n_jobs=-1),
        {"max_depth": [12, 16, 20, None], "min_samples_leaf": [5, 10, 25, 50],
         "max_features": [0.3, 0.5, 0.7], "max_samples": [0.3, 0.5]}, n_iter=6)

### 9.3 SVM (SVR com kernel RBF)

A Máquina de Vetores de Suporte para regressão (SVR) procura uma função que deixe a maior parte dos pontos dentro de uma margem `epsilon`, penalizando com `C` os que ficam fora. O kernel RBF permite relações não lineares.

**Dificuldade prática:** o custo de treino do SVR cresce aproximadamente com o quadrado do número de linhas. Com mais de 340 mil linhas no treino, o ajuste levaria horas. Por isso o SVR é treinado com uma amostra aleatória de `AMOSTRA_SVR` linhas (padrão 30 mil) e a busca de hiperparâmetros usa 8 mil linhas. A avaliação continua no teste inteiro.

In [ ]:
svr = make_pipeline(StandardScaler(), SVR(kernel="rbf", cache_size=1000))
espaco_svr = {"svr__C": [0.3, 1, 3, 10], "svr__epsilon": [0.05, 0.1, 0.2], "svr__gamma": ["scale", 0.01, 0.03]}

# busca numa amostra pequena e refit na amostra maior
idx_svr_busca = rng.choice(len(X_train), size=min(8_000, len(X_train)), replace=False)
t0 = time.time()
busca_svr = RandomizedSearchCV(svr, espaco_svr, n_iter=6, cv=cv, scoring=mae_escala_original,
                               random_state=SEED, n_jobs=-1, refit=False)
busca_svr.fit(X_train.iloc[idx_svr_busca], ylog_train[idx_svr_busca], groups=grupos[idx_svr_busca])
svr.set_params(**busca_svr.best_params_)
print(f"Busca do SVR: {time.time() - t0:.0f} s | melhores: {busca_svr.best_params_}")

ajustar("SVM (SVR, kernel RBF)", "Visto em sala", svr, {}, amostra=AMOSTRA_SVR)
resultados[-1]["MAE validação (CV)"] = -busca_svr.best_score_
resultados[-1]["hiperparâmetros"] = ", ".join(f"{k.split('__')[-1]}={v}" for k, v in busca_svr.best_params_.items()) + f", treino com {AMOSTRA_SVR:,} linhas"

### 9.4 Rede neural (MLP)

Perceptron multicamadas com ativação ReLU, otimizador Adam e **parada antecipada** (`early_stopping`): 10% do treino fica separado e o treino para quando o erro nessa parte deixa de cair por 10 épocas. Testamos arquiteturas com 2 e 3 camadas ocultas, a regularização L2 (`alpha`) e a taxa de aprendizado.

Redes neurais são sensíveis à escala, por isso os atributos passam pelo `StandardScaler` depois do log.

In [ ]:
ajustar("Rede Neural (MLP)", "Visto em sala",
        make_pipeline(StandardScaler(), MLPRegressor(max_iter=200, early_stopping=True, n_iter_no_change=10,
                                                     batch_size=512, random_state=SEED)),
        {"mlpregressor__hidden_layer_sizes": [(64, 32), (128, 64), (128, 64, 32), (256, 128)],
         "mlpregressor__alpha": [1e-5, 1e-4, 1e-3],
         "mlpregressor__learning_rate_init": [5e-4, 1e-3, 3e-3]}, n_iter=6)

mlp = info["Rede Neural (MLP)"]["modelo"][-1]
plt.figure(figsize=(8, 3.6))
plt.plot(mlp.loss_curve_, label="perda no treino")
plt.title(f"Rede neural: curva de aprendizado ({mlp.n_iter_} épocas)")
plt.xlabel("Época"); plt.ylabel("Perda (MSE em log/2)")
plt.legend(); plt.tight_layout(); salvar("02_curva_mlp.png"); plt.show()

### 9.5 Modelos não estudados em sala

| Modelo | Ideia |
|---|---|
| **LightGBM** | *Gradient boosting*: cada nova árvore corrige o erro das anteriores. Cresce as árvores folha a folha e agrupa os valores em histogramas, o que o deixa muito rápido em bases grandes. Foi o modelo mais usado pelos primeiros colocados da competição Optiver |
| **XGBoost** | Também é *gradient boosting*, com regularização explícita nos pesos das folhas e crescimento nível a nível |
| **Extra Trees** | Parecido com o Random Forest, mas sorteia também o ponto de corte de cada divisão. Fica mais aleatório e costuma ter menos variância |

In [ ]:
ajustar("LightGBM", "Não visto em sala",
        lgb.LGBMRegressor(n_estimators=600, random_state=SEED, n_jobs=-1, verbose=-1),
        {"learning_rate": [0.03, 0.05, 0.1], "num_leaves": [31, 63, 127], "min_child_samples": [20, 50, 100],
         "subsample": [0.7, 0.9], "subsample_freq": [1], "colsample_bytree": [0.6, 0.8, 1.0],
         "reg_lambda": [0, 1, 5]}, n_iter=10)

ajustar("XGBoost", "Não visto em sala",
        xgb.XGBRegressor(n_estimators=500, random_state=SEED, n_jobs=-1, tree_method="hist"),
        {"learning_rate": [0.03, 0.05, 0.1], "max_depth": [4, 6, 8], "min_child_weight": [1, 5, 20],
         "subsample": [0.7, 0.9], "colsample_bytree": [0.6, 0.8, 1.0], "reg_lambda": [1, 5]}, n_iter=10)

ajustar("Extra Trees", "Não visto em sala",
        ExtraTreesRegressor(n_estimators=200, random_state=SEED, n_jobs=-1),
        {"max_depth": [16, 20, None], "min_samples_leaf": [5, 10, 25], "max_features": [0.5, 0.7, 1.0],
         "max_samples": [0.3, 0.5], "bootstrap": [True]}, n_iter=6)

### 9.6 Comitê (modelo final candidato)

Juntamos os **três modelos com menor MAE na validação cruzada** numa média simples das previsões em log. A escolha dos três usa a validação, não o teste. Modelos diferentes erram em janelas diferentes, e a média costuma cancelar parte desses erros.

In [ ]:
validacao = {r["modelo"]: r["MAE validação (CV)"] for r in resultados if not np.isnan(r["MAE validação (CV)"])}
TOP3 = sorted(validacao, key=validacao.get)[:3]
print("Modelos do comitê (por MAE na validação):", TOP3)

pred_comite = np.exp(np.mean([np.log(previsoes[n]) for n in TOP3], axis=0))
registrar("Comitê (média dos 3 melhores)", "Modelo final", pred_comite, None, None, " + ".join(TOP3))

## 10. Comparação dos modelos

In [ ]:
tab = pd.DataFrame(resultados).set_index("modelo")
tab = tab.sort_values("MAE")
mae_base = tab.loc[MELHOR_BASELINE, "MAE"]
tab["ganho vs melhor baseline (MAE)"] = (mae_base - tab["MAE"]) / mae_base * 100

cols = ["categoria", "MAE", "RMSE", "R2", "RMSPE", "MAE validação (CV)", "MAE treino",
        "ganho vs melhor baseline (MAE)", "tempo de treino (s)", "hiperparâmetros"]
display(tab[cols].style.format({
    "MAE": "{:.6f}", "RMSE": "{:.6f}", "R2": "{:.4f}", "RMSPE": "{:.4f}", "MAE validação (CV)": "{:.6f}",
    "MAE treino": "{:.6f}", "ganho vs melhor baseline (MAE)": "{:+.1f}%", "tempo de treino (s)": "{:.1f}"},
    na_rep="-").highlight_min(subset=["MAE", "RMSE", "RMSPE"], color="#c6efce").highlight_max(subset=["R2"], color="#c6efce"))

tab[cols].to_csv("resultados_tde04.csv")
print("Tabela salva em resultados_tde04.csv")

In [ ]:
# Gráfico principal: MAE no teste por modelo
t = tab.sort_values("MAE", ascending=False)
fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=True)
for ax, met in zip(axes, ["MAE", "RMSPE"]):
    barras = ax.barh(t.index, t[met], color=[COR[c] for c in t["categoria"]])
    ax.axvline(t.loc[MELHOR_BASELINE, met], color="#333", ls="--", lw=1)
    ax.set_title(f"{met} no teste (menor é melhor)")
    for b, v in zip(barras, t[met]):
        ax.text(v, b.get_y() + b.get_height() / 2, f" {v:.5f}" if met == "MAE" else f" {v:.3f}", va="center", fontsize=8)
    ax.set_xlim(0, t[met].max() * 1.18)
from matplotlib.patches import Patch
axes[0].legend(handles=[Patch(color=v, label=k) for k, v in COR.items()], loc="upper right", fontsize=8, frameon=False)
plt.tight_layout(); salvar("03_comparacao_modelos.png"); plt.show()

In [ ]:
# Treino x validação x teste: quem generaliza
modelos_ajustados = [n for n in tab.index if tab.loc[n, "categoria"] in ("Visto em sala", "Não visto em sala", "Baseline TDE 03")]
d = tab.loc[modelos_ajustados, ["MAE treino", "MAE validação (CV)", "MAE"]].sort_values("MAE")
x = np.arange(len(d)); w = 0.27
plt.figure(figsize=(13, 4.8))
plt.bar(x - w, d["MAE treino"], w, label="treino", color="#B0B7C3")
plt.bar(x, d["MAE validação (CV)"], w, label="validação (CV)", color="#F2CC8F")
plt.bar(x + w, d["MAE"], w, label="teste", color="#2E86AB")
plt.xticks(x, d.index, rotation=30, ha="right")
plt.ylabel("MAE"); plt.title("Treino x validação x teste")
plt.legend(); plt.tight_layout(); salvar("04_treino_validacao_teste.png"); plt.show()

## 11. Leitura dos resultados

A célula abaixo escreve a leitura **a partir dos números desta execução**, para o texto nunca contradizer a tabela.

In [ ]:
cand = tab[tab["categoria"].isin(["Visto em sala", "Não visto em sala", "Modelo final"])]
FINAL = cand["MAE"].idxmin()  # menor MAE no teste entre os modelos do TDE 04
melhor_val = min(validacao, key=validacao.get)
r = tab.loc[FINAL]
mae_persist = tab.loc["Referência: persistência", "MAE"]

print("=== LEITURA DOS RESULTADOS ===\n")
print(f"1) Melhor modelo no teste: {FINAL}")
print(f"   MAE {r['MAE']:.6f} | RMSE {r['RMSE']:.6f} | R² {r['R2']:.4f} | RMSPE {r['RMSPE']:.4f}")
print(f"   Melhor modelo individual na validação cruzada: {melhor_val}\n")
print(f"2) Contra o melhor baseline do TDE 03 ({MELHOR_BASELINE}, MAE {mae_base:.6f}):")
print(f"   o MAE caiu {(mae_base - r['MAE']) / mae_base * 100:.1f}% e o RMSPE foi de "
      f"{tab.loc[MELHOR_BASELINE, 'RMSPE']:.4f} para {r['RMSPE']:.4f}.")
print(f"   Contra a persistência ingênua (MAE {mae_persist:.6f}): {(mae_persist - r['MAE']) / mae_persist * 100:+.1f}%.\n")

print("3) Ranking no teste (MAE):")
for i, (n, row) in enumerate(tab[tab['categoria'] != 'Referência'].iterrows(), 1):
    print(f"   {i:>2}. {n:<32} {row['MAE']:.6f}  ({row['ganho vs melhor baseline (MAE)']:+.1f}% vs baseline)")
print()

print("4) Overfitting (MAE teste / MAE treino):")
for n in modelos_ajustados:
    a, b = tab.loc[n, "MAE treino"], tab.loc[n, "MAE"]
    razao = b / a if a > 1e-9 else np.inf
    leitura = "forte" if razao > 2 else ("moderado" if razao > 1.2 else "baixo")
    print(f"   {n:<32} {razao:5.1f}x  -> overfitting {leitura}" if np.isfinite(razao) else f"   {n:<32} treino com erro zero -> decorou o treino")

resumo = {"modelo_final": FINAL, "melhor_baseline": MELHOR_BASELINE, "comite": TOP3,
          "metricas_final": {k: float(r[k]) for k in ["MAE", "RMSE", "R2", "RMSPE"]},
          "metricas_baseline": {k: float(tab.loc[MELHOR_BASELINE, k]) for k in ["MAE", "RMSE", "R2", "RMSPE"]},
          "linhas_treino": int(len(treino)), "linhas_teste": int(len(teste)), "n_atributos": len(ATRIB)}
json.dump(resumo, open("resumo_tde04.json", "w"), ensure_ascii=False, indent=2)

## 12. Modelo final: previsões e importância dos atributos

In [ ]:
p = previsoes[FINAL]
amostra = rng.choice(len(y_test), size=min(6000, len(y_test)), replace=False)
lim = np.quantile(y_test, 0.995)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, nome in zip(axes, [MELHOR_BASELINE, FINAL]):
    ax.scatter(y_test[amostra], previsoes[nome][amostra], s=4, alpha=0.25,
               color=COR["Baseline TDE 03"] if nome == MELHOR_BASELINE else COR["Modelo final"])
    ax.plot([0, lim], [0, lim], color="crimson", lw=1)
    ax.set_xlim(0, lim); ax.set_ylim(0, lim)
    ax.set_title(f"{nome}\nMAE {mean_absolute_error(y_test, previsoes[nome]):.6f}")
    ax.set_xlabel("Volatilidade real")
axes[0].set_ylabel("Volatilidade prevista")
plt.tight_layout(); salvar("05_real_vs_previsto.png"); plt.show()

In [ ]:
lgbm = info["LightGBM"]["modelo"]
imp = pd.Series(lgbm.booster_.feature_importance("gain"), index=ATRIB)
imp = (imp / imp.sum() * 100).sort_values().tail(15)

plt.figure(figsize=(8, 5.5))
plt.barh(imp.index, imp.values, color=["#E07A5F" if c in ATRIB_NOVOS else "#7F8FA6" for c in imp.index])
plt.xlabel("% do ganho total (LightGBM)")
plt.title("Atributos mais usados (laranja = novos no TDE 04)")
plt.tight_layout(); salvar("06_importancia_atributos.png"); plt.show()

print(f"Participação dos atributos novos no ganho total: "
      f"{(pd.Series(lgbm.booster_.feature_importance('gain'), index=ATRIB)[ATRIB_NOVOS].sum() / lgbm.booster_.feature_importance('gain').sum() * 100):.1f}%")

## 13. Análise de casos de erro

In [ ]:
erros = teste[["stock_id", "time_id", "target"]].copy()
erros["previsto"] = previsoes[FINAL]
erros["erro_abs"] = (erros["target"] - erros["previsto"]).abs()
erros["erro_rel"] = erros["erro_abs"] / erros["target"]
erros["erro"] = erros["previsto"] - erros["target"]
erros["faixa"] = pd.qcut(erros["target"], 10, labels=[f"D{i}" for i in range(1, 11)])

por_faixa = erros.groupby("faixa", observed=True).agg(volatilidade_media=("target", "mean"),
    erro_rel_medio=("erro_rel", "mean"), vies_medio=("erro", "mean"))
display(por_faixa)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
axes[0].bar(por_faixa.index.astype(str), por_faixa["erro_rel_medio"] * 100, color=COR["Modelo final"])
axes[0].set_title("Erro relativo médio por faixa de volatilidade real")
axes[0].set_xlabel("Decil da volatilidade real (D1 = mais calma, D10 = mais volátil)"); axes[0].set_ylabel("%")
axes[1].bar(por_faixa.index.astype(str), por_faixa["vies_medio"],
            color=["#2E86AB" if v >= 0 else "#E07A5F" for v in por_faixa["vies_medio"]])
axes[1].axhline(0, color="black", lw=0.8)
axes[1].set_title("Viés (previsto - real) por faixa")
axes[1].set_xlabel("Decil da volatilidade real")
plt.tight_layout(); salvar("07_erro_por_faixa.png"); plt.show()

In [ ]:
# Ações e janelas com maior erro
por_acao = erros.groupby("stock_id").agg(erro_rel_medio=("erro_rel", "mean"), volatilidade_media=("target", "mean"),
                                         n=("target", "size")).sort_values("erro_rel_medio", ascending=False)
print("Ações com maior erro relativo médio:")
display(por_acao.head(8))

por_janela = erros.groupby("time_id").agg(erro_abs_medio=("erro_abs", "mean"), volatilidade_media=("target", "mean"),
                                          vies=("erro", "mean")).sort_values("erro_abs_medio", ascending=False)
print("Janelas (time_id) com maior erro absoluto médio:")
display(por_janela.head(8))

corr = np.corrcoef(por_janela["volatilidade_media"], por_janela["erro_abs_medio"])[0, 1]
print(f"Correlação entre a volatilidade média da janela e o erro médio da janela: {corr:.2f}")
print(f"Nas 5% piores janelas, o modelo {'subestimou' if por_janela.head(max(1, len(por_janela)//20))['vies'].mean() < 0 else 'superestimou'} "
      f"a volatilidade em média.")

### Como ler a análise de erros

- **Janelas muito calmas (D1).** Como a volatilidade real é pequena, qualquer erro pequeno vira um erro percentual alto. É aqui que o RMSPE mais pune, e foi por isso que treinamos em log.
- **Janelas muito voláteis (D10).** Os picos de mercado vêm de notícias e eventos que não aparecem nos 10 minutos anteriores. O viés por faixa mostra se o modelo tende a **subestimar** esses picos, o comportamento esperado de qualquer modelo que olha só o passado recente.
- **Janelas de estresse de mercado.** Se a correlação entre volatilidade da janela e erro da janela for alta, os maiores erros se concentram nos mesmos momentos para várias ações ao mesmo tempo, o que confirma o achado do TDE 01 de que a volatilidade é um fenômeno de mercado.
- **Ações difíceis.** As ações da tabela acima têm erro relativo maior. Vale olhar se são ações menos líquidas (spread maior), em que poucos negócios bastam para mover o preço.

## 14. Dificuldades encontradas

1. **Tamanho da base.** Os arquivos de book e trades têm cerca de 3 GB. A extração foi feita uma ação por vez, em paralelo, com cache em Parquet para não repetir o processo.
2. **SVM em base grande.** O SVR não escala para centenas de milhares de linhas. Treinamos com amostra, o que limita o desempenho dele em relação aos modelos de árvore.
3. **Tempo de busca.** A busca de hiperparâmetros foi feita numa amostra do treino e com poucas combinações por modelo (busca aleatória). Uma busca maior poderia melhorar um pouco os números, com custo alto de tempo no Colab gratuito.
4. **Vazamento de informação.** Dois cuidados: divisão por `time_id` (a mesma janela nunca está no treino e no teste) e perfil da ação calculado só com as janelas de treino.
5. **Métrica x função de perda.** A competição usa RMSPE, mas a maioria dos modelos minimiza erro quadrático. Treinar em `log(target)` foi a forma simples de aproximar as duas coisas.
6. **Picos de volatilidade.** Eventos de mercado não estão nos dados de entrada, então parte do erro nas janelas extremas não tem como ser eliminada só com esses atributos.

## 15. Conclusão

Partindo do baseline do TDE 03, o TDE 04 trouxe três evoluções: atributos novos (sub-janelas de tempo, segundo nível do book, liquidez, contexto de mercado e perfil da ação), alvo em escala log e ajuste de hiperparâmetros com validação cruzada por `time_id`.

Comparamos 9 modelos ajustados: Ridge, Árvore de Decisão, Random Forest, SVM (SVR), Rede Neural (MLP) e três modelos que não vimos em sala (LightGBM, XGBoost e Extra Trees), além de um comitê com os três melhores na validação. A tabela da seção 10 e a leitura da seção 11 mostram o ganho de cada um em relação ao melhor baseline, no mesmo conjunto de teste.

Para uma próxima etapa: atributos de janelas vizinhas no tempo (o `time_id` não segue ordem cronológica, mas dá para aproximar a ordem pela correlação entre ações), uma rede neural com *embedding* por ação e uma busca de hiperparâmetros maior para o LightGBM.

**Arquivos gerados por este notebook:** `resultados_tde04.csv` (tabela de comparação), `resumo_tde04.json` (números do modelo final) e a pasta `figuras_tde04/` com os gráficos usados nos slides.